Everything overfits so I am dropping this for a while and going to focus on making a proper near perfect dataset

In [46]:
import os
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
MERGED_DATASET_DIR = "merged_dataset"
dinov2 = AutoModel.from_pretrained("facebook/dinov2-small")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [47]:
# ──── Parameters ────
BATCH_SIZE = 16
NUM_WORKERS = 2 # Set to 0 for Windows to avoid multiprocessing issues
NUM_EPOCHS = 15

In [48]:
# ──── Model Definition ────
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes, hidden_size=384):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
        nn.Linear(hidden_size, 256),
        nn.LayerNorm(256),       # stabilizes activations
        nn.ReLU(),
        nn.Dropout(0.5),         # increased from 0.3
        nn.Linear(256, 128),     # extra layer to slow learning
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)

class ActionDataset(ImageFolder):
    """ImageFolder that excludes background from class discovery entirely."""
    def find_classes(self, directory):
        classes, class_to_idx = super().find_classes(directory)
        classes = [c for c in classes if c != "background"]
        class_to_idx = {c: i for i, c in enumerate(classes)}
        return classes, class_to_idx

In [49]:
# ──── Transforms & Augmentation ────
# Train gets augmentation — both real and synthetic
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Val never gets augmentation — only real images
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [50]:
# ──── Dataset ────
# Without Background (Labelless) images
# train_dataset = ActionDataset(
#     root=os.path.join(MERGED_DATASET_DIR, "train"),
#     transform=transform
# )
# val_dataset = ActionDataset(
#     root=os.path.join(MERGED_DATASET_DIR, "val"),
#     transform=transform
# )

# With Background images
train_dataset = datasets.ImageFolder(
    root=os.path.join(MERGED_DATASET_DIR, "train"),
    transform=train_transform,
)
val_dataset = datasets.ImageFolder(
    root=os.path.join(MERGED_DATASET_DIR, "val"),
    transform=val_transform,
)

In [51]:
# ──── Sanity & Info Checks ────
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")
print(f"Class mapping: {train_dataset.class_to_idx}")

# Check class distribution in train and val
train_labels = [label for _, label in train_dataset.samples]
val_labels = [label for _, label in val_dataset.samples]

print("Train distribution:")
for idx, action in enumerate(train_dataset.classes):
    count = train_labels.count(idx)
    pct = count / len(train_labels) * 100
    print(f"  {action}: {count} ({pct:.1f}%)")

print("\nVal distribution:")
for idx, action in enumerate(val_dataset.classes):
    count = val_labels.count(idx)
    pct = count / len(val_labels) * 100
    print(f"  {action}: {count} ({pct:.1f}%)")

# Check for filename overlap between train and val
train_files = set(os.path.basename(p) for p, _ in train_dataset.samples)
val_files = set(os.path.basename(p) for p, _ in val_dataset.samples)

overlap = train_files & val_files
print(f"Overlapping filenames: {len(overlap)}")
if overlap:
    print("Sample overlaps:")
    for f in list(overlap)[:5]:
        print(f"  {f}")

Train samples: 28705
Val samples:   7228
Classes: ['background', 'fanning', 'trophallaxis']
Class mapping: {'background': 0, 'fanning': 1, 'trophallaxis': 2}
Train distribution:
  background: 3000 (10.5%)
  fanning: 12199 (42.5%)
  trophallaxis: 13506 (47.1%)

Val distribution:
  background: 1000 (13.8%)
  fanning: 2904 (40.2%)
  trophallaxis: 3324 (46.0%)
Overlapping filenames: 0


In [ ]:
# ──── DataLoader ────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)
# Sanity check the DataLoader
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
# Verify a batch looks right
images, labels = next(iter(train_loader))
print(f"\nBatch image shape: {images.shape}")   # Should be [16, 3, 224, 224]
print(f"Batch label shape: {labels.shape}")    # Should be [16]
print(f"Label values: {labels}")               # Should be 0s and 1s

In [ ]:
# ──── Model Initialization ────
# load the model and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# number of actions (classes) is determined by the dataset, so initialize model after dataset
num_classes = len(train_dataset.classes)
print(f"Num classes: {num_classes} → {train_dataset.classes}")
model = DINOv2Classifier(dinov2, num_classes=num_classes).to(device)

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-6},
    {"params": model.classifier.parameters(), "lr": 1e-4}
], weight_decay=0.05)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-7
)
criterion = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

# sanity check output shape matches num_classes
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"Output shape: {out.shape}") # should be [1, num_classes]

In [29]:
# ──── Training and Validation Loops ────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, f1

In [ ]:
best_val_f1 = 0.0
# best_val_acc = 0.0 # Keep track of best val accuracy for model checkpointing - maybe F1 in the future?
SAVE_PATH = "best_model_v2.pt" # name for saving the best model checkpoint

for epoch in range(NUM_EPOCHS):
    train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_f1 = validate(model, val_loader, criterion, device)

    scheduler.step()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train → Loss: {train_loss:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   → Loss: {val_loss:.4f} | F1: {val_f1:.4f}")
    print(f"  LR: {scheduler.get_last_lr()[0]:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  ✓ Best model saved (F1: {best_val_f1:.4f})")

    print()

print(f"Training complete. Best val F1: {best_val_f1:.4f}")

In [ ]:
# ──── Evaluation ────
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))

print("Confusion Matrix:")
cm = confusion_matrix(all_labels, all_preds)
print(cm)